# vectortileserver — demo & bridge test

Renders a PMTiles vector layer in `ipyleaflet`, exercising both fixes this package ships:

- **#2 — conversion:** a vector source → PMTiles via `tippecanoe` (when it's on `PATH`), keeping
  every point at every zoom level.
- **#3 — browser reachability:** the `jupyter_loopback` bridge that lets tiles load in sandboxed
  webviews (Voila / SEPAL / VS Code / Colab), where `http://localhost:<port>` is unreachable.

**Setup:** `micromamba env create -f environment.yml` (installs `tippecanoe` + the package), or
`pip install vectortileserver` with `tippecanoe` on your `PATH`. `tippecanoe` is *optional* for this
notebook — without it we synthesize a tiny PMTiles archive so the bridge/render demo still runs.

In [ ]:
import shutil

HAS_TIPPECANOE = shutil.which("tippecanoe") is not None
print(
    "tippecanoe:",
    "found -> convert a GeoJSON (exercises #2)"
    if HAS_TIPPECANOE
    else "not found -> synthesize a .pmtiles (exercises #3 only)",
)

## 1. Get a PMTiles archive

With `tippecanoe` we generate ~2,000 random points carrying a `map_code` category and let
`TileClient` convert them (retaining every point). Without it we write a minimal one-tile archive
directly, so the rest of the notebook still runs. Both helpers are inline so the notebook is
self-contained.

In [ ]:
import gzip
import json
import random
import tempfile
from pathlib import Path

CATEGORIES = [0, 1, 2, 3]
work = Path(tempfile.mkdtemp(prefix="vectortileserver-demo-"))


def write_points(path, count=2000, spread=0.5, seed=0):
    """A GeoJSON FeatureCollection of random points, each with a map_code."""
    rng = random.Random(seed)
    features = [
        {
            "type": "Feature",
            "properties": {"map_code": rng.choice(CATEGORIES)},
            "geometry": {
                "type": "Point",
                "coordinates": [rng.uniform(0, spread), rng.uniform(0, spread)],
            },
        }
        for _ in range(count)
    ]
    path.write_text(json.dumps({"type": "FeatureCollection", "features": features}))
    return path


def write_minimal_pmtiles(path):
    """A valid single-tile PMTiles archive, without shelling out to tippecanoe."""
    import mapbox_vector_tile
    from pmtiles.tile import Compression, TileType
    from pmtiles.writer import Writer

    corners = [(1024, 1024, 0), (3072, 3072, 1), (1024, 3072, 2), (3072, 1024, 3)]
    tile = mapbox_vector_tile.encode(
        [
            {
                "name": "points",
                "features": [
                    {"geometry": f"POINT({x} {y})", "properties": {"map_code": code}}
                    for x, y, code in corners
                ],
            }
        ]
    )
    with open(path, "wb") as f:
        writer = Writer(f)
        writer.write_tile(0, gzip.compress(tile))
        writer.finalize(
            {
                "tile_type": TileType.MVT,
                "tile_compression": Compression.GZIP,
                "internal_compression": Compression.GZIP,
                "min_zoom": 0,
                "max_zoom": 0,
                "min_lon_e7": 0,
                "min_lat_e7": 0,
                "max_lon_e7": int(0.5 * 1e7),
                "max_lat_e7": int(0.5 * 1e7),
            },
            {"vector_layers": [{"id": "points", "fields": {"map_code": "Number"}}]},
        )
    return path

In [ ]:
from vectortileserver import TileClient

if HAS_TIPPECANOE:
    source = write_points(work / "points.geojson", count=2000)
else:
    source = write_minimal_pmtiles(work / "points.pmtiles")

client = TileClient(source, allowed_directories=[work])
print("data source:", source.name)
print("pmtiles    :", client.pmtiles_path)

## 2. Inspect the client

`pmtiles_url` is what the browser fetches. Inside a Jupyter kernel it is a root-relative **proxy
path** (`/vectortileserver-proxy/<port>/…`); outside one it is the raw loopback URL.
`create_leaflet_layer()` auto-installs the `jupyter_loopback` comm bridge for `server_port`, so tiles
still reach the browser in webviews whose origin is not the jupyter-server.

In [ ]:
print("server_port  :", client.server_port)
print("client_prefix:", client.client_prefix)  # None outside a Jupyter kernel
print("pmtiles_url  :", client.pmtiles_url)
print("layers       :", client.list_layers())
print("bounds       :", client.bounds)
print("center       :", client.center)

## 3. Render as a circle layer

The built-in default style targets polygons (fill + outline); points want a `circle` layer. We pass a
custom MapLibre style — colored by `map_code` — straight through `create_leaflet_layer(style=…)`, the
custom-style pass-through case. Scroll to zoom and use the layers control (top-right) to toggle the
point layer; every point stays put at every zoom level.

In [ ]:
from ipyleaflet import LayersControl, Map

layer_id = client.list_layers()[0]
CATEGORY_COLORS = {0: "#e41a1c", 1: "#377eb8", 2: "#4daf4a", 3: "#984ea3"}

circle_color = ["match", ["get", "map_code"]]
for code_value, color in CATEGORY_COLORS.items():
    circle_color += [code_value, color]
circle_color.append("#999999")  # fallback for any other value

style = {
    "version": 8,
    "sources": {
        "pmtiles_source": {"type": "vector", "url": f"pmtiles://{client.pmtiles_url}"}
    },
    "layers": [
        {
            "id": f"{layer_id}-circles",
            "type": "circle",
            "source": "pmtiles_source",
            "source-layer": layer_id,
            "paint": {
                "circle-radius": 5,
                "circle-color": circle_color,
                "circle-stroke-color": "#ffffff",
                "circle-stroke-width": 1,
                "circle-opacity": 0.85,
            },
        }
    ],
}

layer = client.create_leaflet_layer(style=style)
layer.name = f"{layer_id} (PMTiles)"  # label shown in the layers control

m = Map(center=client.center, zoom=9, scroll_wheel_zoom=True)
m.add(layer)
m.add(LayersControl(position="topright"))
m

## 4. Smoke check (headless)

Even without a browser, confirm the server honors HTTP Range requests — the mechanism PMTiles relies
on. This hits the raw loopback server directly (not the proxy path), so it works from the kernel.

In [ ]:
from urllib.parse import quote

import httpx

url = f"{client.server_url}/pmtiles?filePath={quote(str(client.pmtiles_path), safe='/')}"
resp = httpx.get(url, headers={"range": "bytes=0-127"})

print("status       :", resp.status_code, "(expect 206)")
print("content-range:", resp.headers.get("content-range"))
print("bytes        :", len(resp.content))
assert resp.status_code == 206 and len(resp.content) == 128
print("\nRange serving OK.")

## 5. Verify the bridge in a browser

In JupyterLab the map above already loads tiles (same origin). To prove the bridge for **sandboxed
webviews**, serve this notebook with Voila:

```bash
voila examples/demo.ipynb
```

Open it, pan/zoom the map, and watch the browser **Network** tab: tiles should load with **no direct
`127.0.0.1` / `localhost` requests** — every tile travels over the jupyter-server proxy or the
`jupyter_loopback` comm bridge.

To see the contrast, disable the bridge and reload — tiles should then fail to load in a sandboxed
webview:

```bash
VECTORTILESERVER_DISABLE_JUPYTER_LOOPBACK=1 voila examples/demo.ipynb
```